In [ ]:
import pandas as pd
train_data_whole = pd.read_csv('data/train_data.csv') # Jeu de données contenant la fusion de tous les datasets train
test_data = pd.read_csv('data/test_data.csv') # Jeu de données test

products_data = pd.read_csv('data/products_data.csv')
products_data = products_data.drop_duplicates(subset=['product_id'])

# Sélection d'un échantillon de 60000 clients & isolation de l'échantillon test de 20000 clients

household_ids_val= [f"Household_{i}" for i in range(80001, 100001)]

## Echantillon de 60000 clients
train_data = train_data_whole[~train_data_whole["customer_id"].isin(household_ids_val)]
sampled_ids_train = train_data['customer_id'].drop_duplicates().sample(60000,random_state=42)
train_data = train_data[~train_data.customer_id.isin(sampled_ids_train)] # Jeu de données train de 60000 clients

test_data_train = test_data[test_data['customer_id'].isin(train_data['customer_id'].unique())]# Données de test_data relatif aux 60000 clients

## Echantillon de 20000 clients
validate_data = train_data_whole[train_data_whole["customer_id"].isin(household_ids_val)] # Jeu de données test 20000 clients

display("train data",train_data.head(3),"test data",test_data.head(3),"product data",products_data.head(3),"validate data", validate_data)

C:\Users\User\AppData\Local\Temp\ipykernel_17596\153551963.py:7: DtypeWarning: Columns (43) have mixed types. Specify dtype option on import or set low_memory=False.
  products_data = pd.read_csv('data/products_data.csv')


'train data'

,date,transaction_id,customer_id,product_id,has_loyalty_card,store_id,is_promo,quantity,format,order_channel
0,2023-03-06T00:00:00.000000,Transaction_2113977,Household_1956,Product_70955,0,Store_2,0,1.0,DRIVE,MOBILE_APP
1,2022-02-09T00:00:00.000000,Transaction_1686434,Household_1956,Product_204,0,Store_2,0,1.0,DRIVE,MOBILE_APP
2,2023-01-23T00:00:00.000000,Transaction_2032633,Household_1956,Product_10212,0,Store_2,1,2.0,DRIVE,MOBILE_APP


'test data'

,transaction_id,customer_id,product_id
0,Transaction_2024_1,Household_16874,Product_9790
1,Transaction_2024_1,Household_16874,Product_68295
2,Transaction_2024_1,Household_16874,Product_19494


'product data'

,product_id,product_description,department_key,class_key,subclass_key,sector,brand_key,shelf_level1,shelf_level2,shelf_level3,...,alcool,lactose_free,phenylalanine_free,palm_oil_free,ecoscore,produits_du_monde,regional_product,national_brand,first_price_brand,carrefour_brand
0,Product_33508,PET 1L LORINA TONIC ARTISANAL,Department_10,Class_1000,SubClass_10000,PGC,LORINA,Boissons,"Colas, Thés glacés, Sirops et Sodas","Limonades, Limes et Tonics",...,0,0,0,0,NaN,0,0,1,0,0
1,Product_15347,"1,5L LIMO PLANCOET",Department_10,Class_1000,SubClass_10000,PGC,PLANCOET,Boissons,"Colas, Thés glacés, Sirops et Sodas","Limonades, Limes et Tonics",...,0,0,0,0,NaN,0,1,1,0,0
2,Product_80604,"PET 1,5L LIMONADE SIMPL",Department_10,Class_1000,SubClass_10000,PGC,SIMPL,Boissons,"Colas, Thés glacés, Sirops et Sodas","Limonades, Limes et Tonics",...,0,0,0,0,NaN,0,0,0,1,0


### Features engineering

Le tableau ci-dessous contient les principaux types de caractérisques créées dans notre jeu de données.

<table>
  <thead>
    <tr>
      <th>Nom de la colonne</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>transaction_count_cust</td>
      <td>Nombre total de transactions effectuées par le client</td>
    </tr>
    <tr>
      <td>total_quantity_cust</td>
      <td>Quantité totale de produits achetés par le client</td>
    </tr>
    <tr>
      <td>promo_ratio</td>
      <td>Proportion d'achats effectués en promotion par le client</td>
    </tr>
    <tr>
      <td>loyalty_card_usage</td>
      <td>Indique si le client a utilisé une carte de fidélité</td>
    </tr>
    <tr>
      <td>unique_products_cust</td>
      <td>Nombre de produits uniques achetés par le client</td>
    </tr>
    <tr>
      <td>product_popularity</td>
      <td>Nombre de clients ayant acheté le produit</td>
    </tr>
    <tr>
      <td>promo_ratio_onProd</td>
      <td>Proportion d'achats du produit effectués en promotion</td>
    </tr>
    <tr>
      <td>avg_quantity_onProd</td>
      <td>Quantité moyenne achetée par transaction pour le produit</td>
    </tr>
    <tr>
      <td>days_since_last_purchase_onCust</td>
      <td>Nombre de jours depuis le dernier achat du client</td>
    </tr>
    <tr>
      <td>favorite_month</td>
      <td>Mois préféré d'achat du client</td>
    </tr>
    <tr>
      <td>favorite_quarter</td>
      <td>Trimestre préféré d'achat du client</td>
    </tr>
    <tr>
      <td>interaction_count_onProdCust</td>
      <td>Nombre total de transactions du produit par le client</td>
    </tr>
    <tr>
      <td>total_quantity_onProdCust</td>
      <td>Quantité totale du produit achetée par le client</td>
    </tr>
    <tr>
      <td>avg_quantity_onProdCust</td>
      <td>Quantité moyenne achetée par transaction pour le produit par le client</td>
    </tr>
    <tr>
      <td>promo_ratio_onProdCust</td>
      <td>Proportion d'achats en promotion pour le produit par le client</td>
    </tr>
    <tr>
      <td>days_between_purchases_onProdCust</td>
      <td>Durée en jours entre le premier et le dernier achat du produit par le client</td>
    </tr>
    <tr>
      <td>last_quantity</td>
      <td>Quantité du dernier achat du produit par le client</td>
    </tr>
    <tr>
      <td>days_since_last_purchase_onProdCust</td>
      <td>Nombre de jours écoulés depuis le dernier achat du produit par le client</td>
    </tr>
    <tr>
      <td>is_first_transaction_product_2022</td>
      <td>Indique si le produit fait partie de la première transaction de 2022</td>
    </tr>
    <tr>
      <td>is_first_transaction_product_2023</td>
      <td>Indique si le produit fait partie de la première transaction de 2023</td>
    </tr>
    <tr>
      <td>is_first_transaction_category_2022</td>
      <td>Indique si la catégorie fait partie de la première transaction de 2022</td>
    </tr>
    <tr>
      <td>is_first_transaction_category_2023</td>
      <td>Indique si la catégorie fait partie de la première transaction de 2023</td>
    </tr>
    <tr>
      <td>total_transactions_last_three_mounth</td>
      <td>Nombre total de transactions au cours des trois derniers mois de 2023</td>
    </tr>
    <tr>
      <td>total_quantity_last_three_mounth</td>
      <td>Quantité totale achetée au cours des trois derniers mois de 2023</td>
    </tr>
    <tr>
      <td>total_transactions_onDominant</td>
      <td>Nombre total de transactions pour les catégories dominantes</td>
    </tr>
    <tr>
      <td>total_quantity_onDominant</td>
      <td>Quantité totale achetée pour les catégories dominantes</td>
    </tr>
    <tr>
      <td>proportion_transaction_onDominant</td>
      <td>Proportion de transactions sur les catégories dominantes</td>
    </tr>
    <tr>
      <td>ispurchase</td>
      <td>Variable cible indiquant si le produit a été acheté en 2024</td>
    </tr>
    <tr>
      <td>frequence_encoding_columns</td>
      <td>Colonnes représentant les fréquences d'apparition des catégories</td>
    </tr>
  </tbody>
</table>

In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def dataTransform_(data):

    #Jointure avec la base produit
  
    data = pd.merge(data,products_data[['product_id','shelf_level1']],on='product_id',how='left')

    # Conversion des dates en format datetime
    data['date'] = pd.to_datetime(data['date'])

    # Création de l'année pour filtrage ultérieur
    data['year'] = data['date'].dt.year
    ################################################################################
    #1. Variables client
    # A. Nombre total de transactions par client
    client_summary = data.groupby('customer_id').agg(
        transaction_count_cust=('transaction_id', 'nunique'),
        total_quantity_cust=('quantity', 'sum'),
        promo_ratio=('is_promo', 'mean'),
        loyalty_card_usage=('has_loyalty_card', 'max'),  # Assume 1 si carte de fidélité utilisée
        unique_products_cust=('product_id', 'nunique')
    ).reset_index()

    # B. Canaux d'achat préférés
    order_channel_dist = data.groupby(['customer_id', 'order_channel']).size().unstack(fill_value=0)
    order_channel_dist = order_channel_dist.div(order_channel_dist.sum(axis=1), axis=0).reset_index()

    # C. Formats préférés
    format_dist = data.groupby(['customer_id', 'format']).size().unstack(fill_value=0)
    format_dist = format_dist.div(format_dist.sum(axis=1), axis=0).reset_index()

    # Fusion des caractéristiques client
    client_summary = client_summary.merge(order_channel_dist, on='customer_id', how='left')
    client_summary = client_summary.merge(format_dist, on='customer_id', how='left')

    ################################################################################
    # 2. Variables produits
    # A. Popularité d'un produit
    product_summary = data.groupby('product_id').agg(
        product_popularity=('customer_id', 'nunique'),
        promo_ratio_onProd=('is_promo', 'mean'),
        avg_quantity_onProd=('quantity', 'mean')
    ).reset_index()

    #B. Les catégories dominantes
    
    # Liste des catégories dominantes
    domaninatCat = ['Crèmerie et Produits laitiers','Epicerie salée','Epicerie sucrée','Fruits et Légumes',
                    'Charcuterie et Traiteur','Boissons','Entretien et Nettoyage']

    
    # Filtrage des transactions appartenant aux catégories dominantes
    data['is_dominant_shelf_level1'] = data['shelf_level1'].isin(domaninatCat)

    # Calcul des indicateurs pour les catégories dominantes
    dominant_stats = (
        data[data['is_dominant_shelf_level1']]
        .groupby('shelf_level1')
        .agg(
            total_transactions_onDominant=('transaction_id', 'nunique'),
            total_quantity_onDominant=('quantity', 'sum'),
            proportion_transaction_onDominant=('transaction_id', lambda x: len(x) / len(data)),
            is_dominant_shelf_level1=('is_dominant_shelf_level1', 'max')          # Est une catégorie dominante
        )
        .reset_index()
    ) ## delete it

    ################################################################################
    # 3.Variables temporelles
    # A. Temps depuis le dernier achat
    data['days_since_last_purchase_onCust'] = data.sort_values(['customer_id', 'date']).groupby('customer_id')['date'].diff().dt.days
    days_since_last_purchase = data.groupby('customer_id')['days_since_last_purchase_onCust'].min().reset_index()

    # B. Mois préféré par clients
    data['month'] = data['date'].dt.month
    data['quarter'] = data['date'].dt.quarter
    seasonality = data.groupby('customer_id').agg(
        favorite_month=('month', lambda x: x.mode()[0]),  # Mois préféré
        favorite_quarter=('quarter', lambda x: x.mode()[0])  # Trimestre préféré
    ).reset_index()


    ################################################################################
    # 4. Caractéristiques spécifiques à l'interaction client-produit
    # A. Aggrégation générale
    customer_product_summary = data.groupby(['customer_id', 'product_id']).agg(
        interaction_count_onProdCust=('transaction_id', 'count'),   # Nombre d'achats du produit par le client
        total_quantity_onProdCust=('quantity', 'sum'),             # Quantité totale du produit achetée
        avg_quantity_onProdCust=('quantity', 'mean'),              # Quantité moyenne par achat
        promo_ratio_onProdCust=('is_promo', 'mean'),               # Proportion d'achats en promo
        last_purchase_onProdCust=('date', 'max'),                  # Dernière date d'achat
        first_purchase_onProdCust=('date', 'min')                  # Première date d'achat
    ).reset_index()

    #B. Ajout d'une colonne pour le temps écoulé entre le premier et le dernier achat sur un produit spécifique
    customer_product_summary['days_between_purchases_onProdCust'] = (
        customer_product_summary['last_purchase_onProdCust'] - customer_product_summary['first_purchase_onProdCust']
    ).dt.days

    #C. Ajout de la dernière quantité achetée pour chaque produit par client
    latest_transactions = data.sort_values(['customer_id', 'product_id', 'date']).groupby(['customer_id', 'product_id']).last().reset_index()
    latest_transactions = latest_transactions[['customer_id', 'product_id', 'quantity', 'date']]
    latest_transactions.rename(columns={'quantity': 'last_quantity', 'date': 'last_purchase_date'}, inplace=True)

    # Ajout au résumé client-produit
    customer_product_summary = customer_product_summary.merge(
        latest_transactions, on=['customer_id', 'product_id'], how='left'
    )

    # Calcul du nombre de jours écoulés depuis le dernier achat jusqu'à la date de référence
    reference_date = pd.Timestamp('2024-01-04') # Date supposer jour du premier achat

    customer_product_summary['days_since_last_purchase_onProdCust'] = (reference_date - customer_product_summary['last_purchase_date']).dt.days
    customer_product_summary = customer_product_summary.drop(columns=['last_purchase_onProdCust','first_purchase_onProdCust','last_purchase_date'])

    ################################################################################
    # 5. Identifier première transaction n-1 et n-2 clients

    first_transactions = data.groupby(by=['customer_id','year'], as_index=False).agg(
        first_transaction_date=('date', 'min')  # Date de la première transaction
    )

    # Associer la première transaction à ses produits
    first_transaction_products = data.merge(
        first_transactions,
        left_on=['customer_id','year'],
        right_on=['customer_id', 'year'],
        how='inner'
    )

    # Garder uniquement les produits de la première transaction
    first_transaction_products = first_transaction_products[
        first_transaction_products['date'] == first_transaction_products['first_transaction_date']
    ]

    # Créer une feature pour indiquer si le produit fait partie de la première transaction
    first_transaction_set = set(
        zip(
            first_transaction_products['customer_id'],
            first_transaction_products['product_id'],
            first_transaction_products['year']
        )
    )

    first_transaction_categories_set = set(
    zip(
        first_transaction_products['customer_id'],
        first_transaction_products['shelf_level1'], 
        first_transaction_products['year']
    )
    )

    customer_product_summary['is_first_transaction_product_2022'] = customer_product_summary.apply(
    lambda row: 1 if (row['customer_id'], row['product_id'], 2022) in first_transaction_set else 0,
    axis=1
    )
    customer_product_summary['is_first_transaction_product_2023'] = customer_product_summary.apply(
        lambda row: 1 if (row['customer_id'], row['product_id'], 2023) in first_transaction_set else 0,
        axis=1
    )

    ################################################################################
    # 6. Filtrer les données des trois derniers trimestres de 2023
    start_date_q3 = "2023-09-01"
    end_date_q4 = "2023-12-31"

    train_data_last_three_mounth = data[(data['date'] >= start_date_q3) & (data['date'] <= end_date_q4)]

    # Regrouper par client et produit pour calculer les indicateurs
    train_data_three_mounth_stats = (
        train_data_last_three_mounth
        .groupby(['customer_id', 'product_id'])
        .agg(
            total_transactions_last_three_mounth=('transaction_id', 'nunique'),  # Nombre de transactions
            total_quantity_last_three_mounth=('quantity', 'sum')               # Quantité totale achetée
        )
        .reset_index()
    )

    test_data_purchase = set(zip(test_data['customer_id'], test_data['product_id']))

    finalData = data.drop_duplicates(subset=['customer_id','product_id'])[['customer_id','product_id']]


    categorical_columns = [ 'product_description', 'department_key', 'class_key',
       'subclass_key', 'sector', 'brand_key', 'shelf_level1', 'shelf_level2',
       'shelf_level3', 'shelf_level4'] # à ajouter pour frequency encoding ultérieur
    
    finalData = pd.merge(finalData,products_data[categorical_columns + ['product_id']],on='product_id',how='left')

    # Joindre info sur catégorie / produit / client
    finalData['is_first_transaction_category_2022'] = finalData.apply(
    lambda row: 1 if (row['customer_id'], row['shelf_level1'], 2022) in first_transaction_categories_set else 0,
    axis=1
    )
    finalData['is_first_transaction_category_2023'] = finalData.apply(
        lambda row: 1 if (row['customer_id'], row['shelf_level1'], 2023) in first_transaction_categories_set else 0,
        axis=1
    )

    ################################################################################
    ################################################################################
    # Reconstruction du dataset
    finalData = pd.merge(finalData,client_summary,on='customer_id',how='left')
    finalData = pd.merge(finalData,seasonality,on='customer_id',how='left')
    finalData = pd.merge(finalData,product_summary,on='product_id',how='left')
    finalData = pd.merge(finalData,customer_product_summary,on=['customer_id','product_id'],how='left')
    finalData = pd.merge(finalData,train_data_three_mounth_stats, on=['customer_id', 'product_id'], how='left')
    finalData = pd.merge(finalData,dominant_stats, on='shelf_level1', how='left')


    # Remplir les valeurs manquantes pour les clients/produits hors période
    finalData[['total_transactions_last_three_mounth', 'total_quantity_last_three_mounth','is_dominant_shelf_level1']] = finalData[['total_transactions_last_three_mounth', 'total_quantity_last_three_mounth','is_dominant_shelf_level1']].fillna(0)
    
    # Remplissage des valeurs non-dominantes
    finalData[['total_transactions_onDominant', 'total_quantity_onDominant', 'proportion_transaction_onDominant']] = finalData[
        ['total_transactions_onDominant', 'total_quantity_onDominant', 'proportion_transaction_onDominant']
    ].fillna(0)

    # Conversion en binaire de la variable `is_dominant_shelf_level1`
    finalData['is_dominant_shelf_level1'] = finalData['is_dominant_shelf_level1'].astype(int)

    # Création de la colonne target 'ispurchase'
    finalData['ispurchase'] = finalData.apply(
        lambda row: 1 if (row['customer_id'], row['product_id']) in test_data_purchase else 0, axis=1
    )

    categorical_columns = [ 'product_description', 'department_key', 'class_key',
       'subclass_key', 'sector', 'brand_key', 'shelf_level1', 'shelf_level2',
       'shelf_level3', 'shelf_level4']

    ################################################################################
    #7. Frequency encoding des caractérisque produit sur le dataset résumé
    # Remplacer chaque catégorie par sa fréquence d'apparition (frequence encoding)
    for col in categorical_columns:
        freq_map = finalData[col].value_counts(normalize=True)  # Calcul des fréquences
        finalData[f'{col}_frequency'] = finalData[col].map(freq_map).fillna(0)  # Remplacement par fréquence


    return finalData

## Modelisation

In [22]:
# Transformation des datasets train & validation
finalData = dataTransform_(train_data)
data_validate = dataTransform_(validate_data)

In [2]:
# Identifier les colonnes communes aux deux DataFrames
gather_col = [col for col in finalData.columns if col in data_validate.columns]

# Filtrer les deux DataFrames pour ne conserver que les colonnes communes
finalData = finalData[gather_col]
data_validate = data_validate[gather_col]

print("Colonnes finales communes :", gather_col)

Colonnes finales communes : ['customer_id', 'product_id', 'product_description', 'department_key', 'class_key', 'subclass_key', 'sector', 'brand_key', 'shelf_level1', 'shelf_level2', 'shelf_level3', 'shelf_level4', 'is_first_transaction_category_2022', 'is_first_transaction_category_2023', 'transaction_count_cust', 'total_quantity_cust', 'promo_ratio', 'loyalty_card_usage', 'unique_products_cust', 'ANNULEE', 'ARRIVEE', 'EN COURS DE PREPARATION', 'EXPEDIEE', 'LIVREE', 'MOBILE_APP', 'NON LIVREE', 'PREPAREE', 'PRISE EN COMPTE', 'WEBSITE', 'CLCV', 'DRIVE', 'LEX', 'favorite_month', 'favorite_quarter', 'product_popularity', 'promo_ratio_onProd', 'avg_quantity_onProd', 'interaction_count_onProdCust', 'total_quantity_onProdCust', 'avg_quantity_onProdCust', 'promo_ratio_onProdCust', 'days_between_purchases_onProdCust', 'last_quantity', 'days_since_last_purchase_onProdCust', 'is_first_transaction_product_2022', 'is_first_transaction_product_2023', 'total_transactions_last_three_mounth', 'total_q

#### Training

In [26]:
# Colonnes catégorielles pour suppression avant entraînement
categorical_columns = ['customer_id','product_id','shelf_level1','product_description', 'department_key', 'class_key',
        'subclass_key', 'sector', 'brand_key', 'shelf_level1', 'shelf_level2',
        'shelf_level3', 'shelf_level4','ispurchase'] 

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y_plus = finalData['ispurchase']

X_plus = finalData.drop(columns=categorical_columns) # Suppression des colonnes catégorielles

scaler = StandardScaler()
X_plus = scaler.fit_transform(X_plus)

# Diviser en jeu d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X_plus, y_plus, test_size=0.2, stratify=y_plus,random_state=42)

In [6]:
# Hitrate@10 evaluation function

def hitrate_at_k(true_data: pd.DataFrame,
                 predicted_data: pd.DataFrame,
                 k: int = 10) -> float:
    """
    This function calculates the hitrate at k for the recommendations.
    It assesses how relevant our 10 product recommendations are.
    In other words, it calculates the proportion of recommended products that are actually purchased by the customer.

    Args:
        true_data: a pandas DataFrame containing the true data
            customer_id: the customer identifier
            product_id: the product identifier that was purchased in the test set
        predicted_data: a pandas DataFrame containing the predicted data
            customer_id: the customer identifier
            product_id: the product identifier that was recommended
            rank: the rank of the recommendation. the rank should be between 1 and 10.
        k: the number of recommendations to consider. k should be between 1 and 10.
    
    Returns:
        The hitrate at k
    """
    
    data = pd.merge(left = true_data, right = predicted_data, how = "left", on = ["customer_id", "product_id"])
    df = data[data["rank"] <= k]
    non_null_counts = df.groupby('customer_id')['rank'].apply(lambda x: x.notna().sum()).reset_index(name='non_null_count')
    distinct_products_per_customer = data.groupby('customer_id')['product_id'].nunique().reset_index(name='distinct_product_count')
    df = pd.merge(left = distinct_products_per_customer, right = non_null_counts, how = "left", on = "customer_id")
    df["denominator"] = [min(df.iloc[i].distinct_product_count,k) for i in range(len(df))]
    df = df.fillna(0)
    return (df["non_null_count"]/df["denominator"]).mean()

def getHitRate(data,data_test):
    # Rank the frequency for each customer in descending order
    data.loc[:,'rank'] = data.groupby('customer_id')['proba'].rank(ascending=False, method="first").astype("Int64").copy()
    top_10_recommendations = data[data['rank'] <= 10].copy()
    top_10_recommendations = top_10_recommendations[['customer_id','product_id','rank']]
    # Calculate the hitrate at k for k = 10
    frequency_model_hitrate_at_10 = hitrate_at_k(data_test,top_10_recommendations,10)
    return frequency_model_hitrate_at_10

from sklearn.metrics import roc_auc_score, accuracy_score,recall_score,precision_score,f1_score,classification_report
def evalutionMetrics(y_test, y_pred,y_pred_prob):
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc_ = roc_auc_score(y_test, y_pred_prob)

    # Affichage des métriques
    print("AUC-ROC: ",roc_auc_ )
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

### DATA MODELING

In [7]:
#Predictive Models
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


models = [
          LogisticRegression(random_state=42),
          CatBoostClassifier(task_type="CPU", logging_level='Silent'),
          LGBMClassifier(device='cpu'),
          XGBClassifier(tree_method='hist')
          ]

In [8]:
def train_models(model, X_train, X_test, y_train, y_test,X_plus,X_Final):
    model.fit(X_train, y_train)
    
    scaler = StandardScaler()
    X_plus = scaler.fit_transform(X_plus)
    X_Final.loc[:,'proba'] = model.predict_proba(X_plus)[:,1]
    output = {
      'AUC-Train': roc_auc_score(y_train, model.predict_proba(X_train)[:,1]),
      'AUC-Test': roc_auc_score(y_test, model.predict_proba(X_test)[:,1]),
      'Accuracy': accuracy_score(y_test, model.predict(X_test)),
      'Precision': precision_score(y_test, model.predict(X_test)),
      'Recall': recall_score(y_test, model.predict(X_test)),
      'F1': f1_score(y_test, model.predict(X_test)),
      'HitRate10': getHitRate(X_Final,test_data_train)
      }
    
    
    return output

In [13]:
%%time
import time

name = []
aucTrain = []
aucTest = []
accuracy = []
precision = []
recall = []
f1 = []
time_ = []
hitRate =[]

for model in models:
    start = time.time()
    results = train_models(model, X_train, X_test, y_train, y_test,X_plus,finalData)

    name.append(type(model).__name__)
    aucTrain.append(results['AUC-Train'])
    aucTest.append(results['AUC-Test'])
    accuracy.append(results['Accuracy'])
    precision.append(results['Precision'])
    recall.append(results['Recall'])
    f1.append(results['F1'])
    hitRate.append(results['HitRate10'])
    time_.append(time.time()-start)

#Initialise data of lists
base_models = pd.DataFrame(data=[name, aucTrain,aucTest, accuracy, precision, recall, f1,hitRate ,time_]).T
base_models.columns = ['Model', 'AUC-Train','AUC-Test', 'Accuracy', 'Precision', 'Recall', 'F1','HitRate10', 'Time']
base_models.sort_values('AUC-Test', ascending=False, inplace=True)

base_models # Affichage des métriques

hitRate  0.34600652513144686
hitRate  0.4781196887020306
[LightGBM] [Info] Number of positive: 38582, number of negative: 1033609
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.193675 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5931
[LightGBM] [Info] Number of data points in the train set: 1072191, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035984 -> initscore=-3.288026
[LightGBM] [Info] Start training from score -3.288026
hitRate  0.3893951705081949
hitRate  0.4249877001520708
CPU times: total: 29min 13s
Wall time: 12min 49s


### Modèle LigthGBM

In [16]:
### LigthGBM
import lightgbm as lgb

# Conversion des données en format LightGBM Dataset
train_data_ = lgb.Dataset(X_train, label=y_train)
test_data_ = lgb.Dataset(X_test, label=y_test, reference=train_data_)

# Définition des paramètres LightGBM
params = {
    "objective": "binary",         # Tâche de classification binaire
    "metric": "binary_logloss",    # Métrique d'évaluation
    "boosting_type": "gbdt",       # Gradient Boosting Decision Tree
    "num_leaves": 80,              # Nombre maximum de feuilles dans chaque arbre
    "learning_rate": 0.05,          # Taux d'apprentissage
    "feature_fraction": 0.5,       # Fraction des caractéristiques à utiliser
    "seed": 42,                     # Reproductibilité
}

num_round = 200 # Nombre d'arbre dans le modèle
model_gbm = lgb.train(
    params,
    train_data_,
    num_boost_round=num_round,
    valid_sets=[train_data_, test_data_],  # Validation sur l'ensemble de test
    valid_names=["train", "test"],       # Noms des ensembles de validation
)

[LightGBM] [Info] Number of positive: 38582, number of negative: 1033609
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.143225 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5917
[LightGBM] [Info] Number of data points in the train set: 1072191, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035984 -> initscore=-3.288026
[LightGBM] [Info] Start training from score -3.288026


## SUMISSION FILE

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [5]:
import lightgbm as lgb

X_plus_validate = data_validate.drop(columns= categorical_columns) # For test
X_plus_validate = scaler.fit_transform(X_plus_validate)
proba_test = model_gbm.predict(X_plus_validate, num_iteration=model_gbm.best_iteration)
data_validate.loc[:,'proba']=proba_test.tolist()

In [7]:
data_validate.loc[:,'rank'] = data_validate.groupby('customer_id')['proba'].rank(ascending=False, method="first").astype("Int64")
top_10_recommendations = data_validate[data_validate['rank'] <= 10]
top_10_recommendations = top_10_recommendations[['customer_id','product_id','rank']]

# # Create submission file for 

# Keep only the top 10 recommendations for Households between 80001 and 100000
prediction = top_10_recommendations[
    top_10_recommendations.customer_id.isin(
            [
                f"Household_{i}" for i in range(80001,100001)
            ]
        )
    ]

# Print the solution
prediction.head()

,customer_id,product_id,rank
3,Household_80212,Product_57942,1
9,Household_80212,Product_75713,4
13,Household_82655,Product_60499,9
16,Household_83691,Product_56651,5
20,Household_83691,Product_37408,4


In [8]:
def process_and_format_prediction(df):
    # Remplacement des caractères invalides dans les noms de colonnes
    df.columns = df.columns.str.replace('+AF8-', '_', regex=False)
    df = df.replace(r'\+AF8-', '_', regex=True)

    # Nettoyage des colonnes 'customer_id', 'product_id', et 'transaction_id'
    if 'customer_id' in df.columns and df['customer_id'].dtype == 'object':
        df['customer_id'] = df['customer_id'].str.extract('(\d+)').fillna(11).astype(int)
    if 'product_id' in df.columns and df['product_id'].dtype == 'object':
        df['product_id'] = df['product_id'].str.extract('(\d+)').fillna(11).astype(int)
    if 'transaction_id' in df.columns and df['transaction_id'].dtype == 'object':
        df['transaction_id'] = df['transaction_id'].str.replace(r'\D', '', regex=True).fillna(11).astype(int)

    df['id'] = df.index
    df = df[['id'] + [col for col in df.columns if col != 'id']]

    if 'customer_id' not in df.columns or 'product_id' not in df.columns:
        raise ValueError("true_data must contain 'customer_id' and 'product_id' columns")

    # Grouper par customer_id et concaténer les valeurs des produits et des ranks
    prediction_grouped = df.groupby('customer_id').agg({
        'id': 'first',  # Prend la première valeur de 'id'
        'product_id': lambda x: ','.join(map(str, x)),  # Concatène les product_id en chaîne de caractères
        'rank': lambda x: ','.join(map(str, x))  # Concatène les ranks en chaîne de caractères
    }).reset_index()

    # Supprimer la colonne 'id' si elle existe
    if 'id' in prediction_grouped.columns:
        prediction_grouped = prediction_grouped.drop(columns=['id'])

    # Filtrer les données
    prediction_grouped = prediction_grouped[prediction_grouped['customer_id'] != 11]
    prediction_grouped.insert(0, 'id', range(len(prediction_grouped)))
    
       # Vérification des rangs et des doublons
    for index, row in prediction_grouped.iterrows():
        # Vérifier les ranks
        ranks = list(map(int, row['rank'].split(',')))
        if sorted(ranks) != list(range(1, 11)):  # Vérifie que les rangs sont distincts de 1 à 10
            print("Doublon détecté. Les rangs doivent être distincts (de 1 à 10) pour chacun des 10 produits prédits pour un client.\n")
            return None
        # Vérifier les doublons de produits
        products = row['product_id'].split(',')
        if len(products) != len(set(products)):  # Si des doublons sont présents dans les produits
            print("Doublon détecté. Il doit y avoir 10 produits différents par client.\n")
            return None
        

    return prediction_grouped
prediction_grouped=process_and_format_prediction(prediction)
print(prediction_grouped)


<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
C:\Users\User\AppData\Local\Temp\ipykernel_1232\3920815098.py:8: SyntaxWarning: invalid escape sequence '\d'
  df['customer_id'] = df['customer_id'].str.extract('(\d+)').fillna(11).astype(int)
C:\Users\User\AppData\Local\Temp\ipykernel_1232\3920815098.py:10: SyntaxWarning: invalid escape sequence '\d'
  df['product_id'] = df['product_id'].str.extract('(\d+)').fillna(11).astype(int)


          id  customer_id                                         product_id  \
0          0        80001  77284,71445,8449,892,81457,80487,69168,55053,1...   
1          1        80002  42748,23144,445,5969,80799,56807,70952,21842,2...   
2          2        80003  50250,4700,79974,13107,47976,23971,67368,82254...   
3          3        80004  35439,9790,50497,60402,20421,79974,60476,74908...   
4          4        80005  8582,379,65281,30001,54653,36883,72457,59618,7...   
...      ...          ...                                                ...   
19995  19995        99996  62103,9580,77951,13055,39751,20421,59196,9790,...   
19996  19996        99997  62032,5156,54965,45518,68843,25151,39654,27445...   
19997  19997        99998  40951,72217,81578,72686,8582,39893,70197,8229,...   
19998  19998        99999  18088,81947,46171,27193,27966,4078,7115,43071,...   
19999  19999       100000  71540,5237,31642,472,64949,76630,23631,55418,2...   

                       rank  
0      2,

In [9]:
# Create a .csv file to submit on kaggle
# A lancer en local sur votre ordinateur
prediction_grouped.to_csv('submission_list08012025.csv', index=False) ####